# 3D Simulation Visualization

This notebook loads a simulation pickle file and generates a 3D animated GIF of the solar system evolution.

In [1]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from mpl_toolkits.mplot3d import Axes3D
from amuse.units import units
from IPython.display import HTML

/Users/zoutongshen/miniforge3/envs/Amuse-env/lib/python3.12/site-packages/amuse/support/options.py:11: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
# Configuration
input_file = 'results/pkl/example_migration_50kyr.pkl'  # Path relative to where notebook is running
output_file = 'simulation_3d.gif'
fps = 10
step = 1  # Skip frames for faster generation

In [3]:
def load_data(filename):
    print(f"Loading {filename}...")
    with open(filename, 'rb') as f:
        data = pickle.load(f)
    print(f"Loaded {len(data['times'])} snapshots.")
    return data

def update(frame, data, scat_planets, scat_planetesimals, title_text):
    # Data for this frame
    massive = data['snapshots_massive'][frame]
    planetesimals = data['snapshots_planetesimals'][frame]
    time = data['times'][frame]
    
    # Update title
    title_text.set_text(f'Time: {time:.1f} yr')
    
    # Update massive bodies
    x_mass = massive.x.value_in(units.AU)
    y_mass = massive.y.value_in(units.AU)
    z_mass = massive.z.value_in(units.AU)
    
    scat_planets._offsets3d = (x_mass, y_mass, z_mass)
    
    # Update planetesimals
    if len(planetesimals) > 0:
        x_plan = planetesimals.x.value_in(units.AU)
        y_plan = planetesimals.y.value_in(units.AU)
        z_plan = planetesimals.z.value_in(units.AU)
        scat_planetesimals._offsets3d = (x_plan, y_plan, z_plan)
        
    return scat_planets, scat_planetesimals, title_text

In [4]:
# Load Data
data = load_data(input_file)
times = data['times']

Loading results/pkl/example_migration_50kyr.pkl...
Loaded 101 snapshots.


In [5]:
# Setup 3D Plot
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection='3d')

# Initial data for limits and style
massive_0 = data['snapshots_massive'][0]

# Determine axis limits
max_range = np.max(np.abs(massive_0.position.value_in(units.AU))) * 1.5
max_range = max(max_range, 6.0) # Minimal size

ax.set_xlim(-max_range, max_range)
ax.set_ylim(-max_range, max_range)
ax.set_zlim(-max_range, max_range)

ax.set_xlabel('X [AU]')
ax.set_ylabel('Y [AU]')
ax.set_zlabel('Z [AU]')

# Define colors for massive bodies
colors = ['yellow'] + ['blue'] * (len(massive_0) - 1)
names = getattr(massive_0, 'name', [])
if len(names) > 0:
    colors = []
    for name in names:
        if name == 'Sun': colors.append('yellow')
        elif name == 'Jupiter': colors.append('brown')
        elif name == 'Earth': colors.append('blue')
        elif name == 'Mars': colors.append('red')
        else: colors.append('black')

# Initialize scatter plots
x_init = massive_0.x.value_in(units.AU)
y_init = massive_0.y.value_in(units.AU)
z_init = massive_0.z.value_in(units.AU)

scat_planets = ax.scatter(x_init, y_init, z_init, c=colors, s=100, depthshade=False, label='Massive Bodies')
scat_planetesimals = ax.scatter([], [], [], c='gray', s=2, alpha=0.5, label='Planetesimals')
title_text = ax.text2D(0.05, 0.95, "", transform=ax.transAxes)
ax.legend()

# Create Animation
frames = range(0, len(times), step)
anim = FuncAnimation(
    fig, update, frames=frames,
    fargs=(data, scat_planets, scat_planetesimals, title_text),
    blit=False
)

# Save to GIF
print(f"Saving animation to {output_file}...")
anim.save(output_file, writer=PillowWriter(fps=fps))
print("Done!")

plt.close() # Don't display static plot

Saving animation to simulation_3d.gif...
Done!


In [6]:
# Display Animation in Notebook (Javascript)
HTML(anim.to_jshtml())

Animation size has reached 21057199 bytes, exceeding the limit of 20971520.0. If you're sure you want a larger animation embedded, set the animation.embed_limit rc parameter to a larger value (in MB). This and further frames will be dropped.
